In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from itertools import product

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"last12weeks_tempextreme_output\last12weeks_rabi_rice_telangana_weekly_extremetemp.csv")  # replace with your file
df['YEAR'] = df['YEAR'].astype(int)
df = df.sort_values('YEAR')
df = df.set_index('YEAR')
print(df.head())

In [ ]:
# Define target and exogenous columns
target_col = 'Yield'
exog_cols = [col for col in df.columns if col != target_col]

In [ ]:
# Train-test split
train = df.iloc[:-3]
test = df.iloc[-3:]
y_train, y_test = train[target_col], test[target_col]
X_train, X_test = train[exog_cols], test[exog_cols]                                                                                                                                                                     

# Assuming df is your original DataFrame
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['Yield'], marker='o', linestyle='-', color='teal')
plt.title("Yield Over Years", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Yield")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Step 1: Stationarity Tests ----------
def check_stationarity(y, series_name="Series", regression_type='c'):
    print(f"\n=== Stationarity Tests for: {series_name} ===")

    # ADF Test
    adf_result = adfuller(y, autolag='AIC')
    adf_stat, adf_p = adf_result[0], adf_result[1]
    adf_crit = adf_result[4]
    
    # KPSS Test
    kpss_result = kpss(y, regression=regression_type, nlags="auto")
    kpss_stat, kpss_p = kpss_result[0], kpss_result[1]
    kpss_crit = kpss_result[3]
    
    # Summary Table
    summary = pd.DataFrame({
        'Test': ['ADF', 'KPSS'],
        'Test Statistic': [adf_stat, kpss_stat],
        'p-value': [adf_p, kpss_p],
        'Stationary (at 0.05)?': ['Yes' if adf_p < 0.05 else 'No',
                                  'No' if kpss_p < 0.05 else 'Yes']
    })
    print(summary.to_string(index=False))
    
    # Interpretation
    if adf_p < 0.05 and kpss_p >= 0.05:
        print("\n Both tests suggest the series IS stationary.")
    elif adf_p >= 0.05 and kpss_p < 0.05:
        print("\n Both tests suggest the series is NOT stationary. ➤ Consider differencing.")
    else:
        print("\n Mixed results. You may still try differencing or visual inspection.")

In [ ]:
diff_1 = y_train.diff().dropna()
diff_2=diff_1.diff().dropna()
diff_3=diff_2.diff().dropna()
diff_4=diff_3.diff().dropna()
diff_5=diff_4.diff().dropna()
diff_6=diff_5.diff().dropna()
check_stationarity(diff_3, series_name="Yield (Train)", regression_type='c')

In [ ]:
# ---------- Step 2: ACF & PACF ----------
plot_acf(y_train)
plt.title("ACF - Yield")
plt.show()

plot_pacf(y_train)
plt.title("PACF - Yield")
plt.show()

In [ ]:
# ---------- Step 3: Grid Search ARIMAX ----------
def get_best_pdq(y, X):
    p = q = range(0, 3)
    d=range(3,4)
    best_models = []
    for order in product(p, d, q):
        try:
            model = SARIMAX(y, exog=X, order=order, enforce_stationarity=False, enforce_invertibility=False)
            result = model.fit(disp=False)
            best_models.append((order, result.aic, result.bic))
        except:
            continue
    results_df = pd.DataFrame(best_models, columns=['Order', 'AIC', 'BIC']).sort_values(by='AIC')
    return results_df

print("\nFinding best ARIMAX order...")
results_df = get_best_pdq(y_train, X_train)
print(results_df.head())

best_order = results_df.iloc[0]['Order']
print(f"\nBest order: {best_order}")


In [ ]:
# ---------- Step 4: Fit ARIMAX Model ----------
model = SARIMAX(y_train, exog=X_train, order=best_order, enforce_stationarity=False, enforce_invertibility=False)
result = model.fit()
print(result.summary())

In [ ]:
# ---------- Step 5: Drop Insignificant Variables ----------
insig_vars = result.pvalues[result.pvalues > 0.05].index
insig_vars = [v for v in insig_vars if "exog" in v]
drop_cols = [X_train.columns[int(v.split('.')[1])] for v in insig_vars]
X_train_reduced = X_train.drop(columns=drop_cols)
X_test_reduced = X_test.drop(columns=drop_cols)
print(f"\nDropping insignificant columns: {drop_cols}")

In [ ]:
# ---------- Step 6: Refit ARIMAX ----------
model_refit = SARIMAX(y_train, exog=X_train_reduced, order=best_order, enforce_stationarity=False, enforce_invertibility=False)
result_refit = model_refit.fit()
print(result_refit.summary())

In [ ]:
# ---------- Step 7: Predict and Evaluate ----------
forecast = result_refit.get_forecast(steps=len(y_test), exog=X_test_reduced)
pred_mean = forecast.predicted_mean
print(pred_mean)


def evaluate(y_true, y_pred, label=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"\n Evaluation ({label}):")
    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%")
    return mae, rmse, mape

evaluate(y_test, pred_mean, label="ARIMAX (original)")

In [ ]:
# Ensure x-axis values are integers (years)
x_years = y_test.index.astype(int)  # Convert index to integer years

plt.figure(figsize=(10, 5))
plt.plot(x_years, y_test.values, marker='o', label='Actual', linewidth=2)
plt.plot(x_years, pred_mean.values, marker='x', linestyle='--', label='Predicted', linewidth=2, color='orange')

# Optional confidence interval (if available)
if 'conf_int' in locals():
    plt.fill_between(x_years, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                     color='gray', alpha=0.3, label='95% CI')

plt.title("Actual vs Predicted Yield (Test Set)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Yield")
plt.xticks(ticks=x_years, labels=[str(int(y)) for y in x_years])  # Force integer labels
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Step 9: Repeat with Normalized Data ----------
scaler_y = StandardScaler()
scaler_x = StandardScaler()

y_train_norm = scaler_y.fit_transform(y_train.values.reshape(-1,1)).flatten()
y_test_norm = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()

X_train_norm = pd.DataFrame(scaler_x.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_norm = pd.DataFrame(scaler_x.transform(X_test), columns=X_test.columns, index=X_test.index)

results_df_norm = get_best_pdq(y_train_norm, X_train_norm)
best_order_norm = results_df_norm.iloc[0]['Order']
print(f"\nBest order (normalized): {best_order_norm}")

model_norm = SARIMAX(y_train_norm, exog=X_train_norm, order=best_order_norm, enforce_stationarity=False, enforce_invertibility=False)
result_norm = model_norm.fit()
print(result_norm.summary())

# Drop insignificant vars from normalized
insig_vars_norm = result_norm.pvalues[result_norm.pvalues > 0.05].index
drop_cols_norm = [X_train.columns[int(v.split('.')[1])] for v in insig_vars_norm if 'exog' in v]
X_train_norm_reduced = X_train_norm.drop(columns=drop_cols_norm)
X_test_norm_reduced = X_test_norm.drop(columns=drop_cols_norm)

# Refit
model_refit_norm = SARIMAX(y_train_norm, exog=X_train_norm_reduced, order=best_order_norm, enforce_stationarity=False, enforce_invertibility=False)
result_refit_norm = model_refit_norm.fit()

# Forecast using normalized model
pred_norm = result_refit_norm.get_forecast(steps=len(y_test_norm), exog=X_test_norm_reduced).predicted_mean

# Inverse transform the predictions
pred_norm_rescaled = scaler_y.inverse_transform(pred_norm.values.reshape(-1, 1)).flatten()

# Evaluate normalized
evaluate(y_test, pred_norm_rescaled, label="ARIMAX (normalized)")

# Compare plot
plt.plot(y_test.index, y_test, label='Actual', marker='o')
plt.plot(y_test.index, pred_norm_rescaled, label='Predicted (Norm)', marker='x')
plt.title("Normalized ARIMAX Prediction vs Actual")
plt.legend()
plt.grid()
plt.show()

In [ ]:
print(pred_mean)